In [200]:
import pandas as pd
import numpy as np
import re

In [202]:
df = pd.read_csv("C:/Users/Jernie/OneDrive/Desktop/GITHUB PROJECTS/EDUCATION DATASET/Student Mental health.csv")

In [204]:
df

,Timestamp,Choose your gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...
96,13/07/2020 19:56:49,Female,21.0,BCS,year 1,3.50 - 4.00,No,No,Yes,No,No
97,13/07/2020 21:21:42,Male,18.0,Engineering,Year 2,3.00 - 3.49,No,Yes,Yes,No,No
98,13/07/2020 21:22:56,Female,19.0,Nursing,Year 3,3.50 - 4.00,Yes,Yes,No,Yes,No
99,13/07/2020 21:23:57,Female,23.0,Pendidikan Islam,year 4,3.50 - 4.00,No,No,No,No,No


## Data Preprocessing

In [208]:
# Inspect unique values and counts for the course column
course_counts = df["What is your course?"].astype(str).str.strip().value_counts(dropna=False).reset_index()
course_counts.columns = ["CourseRaw", "Count"]
course_counts.head(30)

,CourseRaw,Count
0,BCS,18
1,Engineering,17
2,BIT,10
3,Biomedical science,4
4,KOE,4
5,psychology,2
6,Engine,2
7,Pendidikan Islam,2
8,BENL,2
9,Laws,2


In [210]:
# Exact source column name
course_col = "What is your course?"

#Merging near-duplicates (engin/engineering, law/laws, etc.)
def clean_course(s):
    if pd.isna(s): 
        return np.nan
    t = str(s).strip().lower()
    t_norm = re.sub(r'[^a-z]+', ' ', t).strip()   # remove punctuation, unify spacing

    # Exact/near-exact merges
    if t_norm in {"engin", "engine", "engineering"}:
        return "Engineering"
    if t_norm in {"law", "laws"}:
        return "Law"
    if t_norm in {"econs", "economics"}:
        return "Economics"
    if t_norm in {"pendidikan islam", "islamic education"}:
        return "Islamic Education"
    if t_norm.startswith("fiqh"):
        return "Fiqh & Fatwa"
    if t_norm in {"biomedical science", "biomedical sciences"}:
        return "Biomedical Science"
    if t_norm in {"psychology"}:
        return "Psychology"
    if t_norm in {"communication", "communications"}:
        return "Communication"

    # Common acronyms → uppercase
    acr = {
        "bcs":"BCS", "bit":"BIT", "it":"IT", "koe":"KOE", "mhsc":"MHSC",
        "cts":"CTS", "benl":"BENL", "kirkhs":"KIRKHS", "kop":"KOP"
    }
    if t_norm in acr:
        return acr[t_norm]

    # Fallback: title-case words
    return " ".join(w.capitalize() for w in t_norm.split())

# Create the standardized column
df["Course_Standardized"] = df[course_col].apply(clean_course)

# Mappings and counts
mapping = (
    df[[course_col, "Course_Standardized"]]
    .value_counts()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)
print(mapping.head(20))

after_counts = (
    df["Course_Standardized"]
    .value_counts()
    .rename_axis("Course")
    .reset_index(name="Count")
)
print(after_counts.head(20))

       What is your course?      Course_Standardized  Count
0                       BCS                      BCS     18
1               Engineering              Engineering     17
2                       BIT                      BIT     10
3        Biomedical science       Biomedical Science      4
4                       KOE                      KOE      4
7                      BENL                     BENL      2
8                      Laws                      Law      2
6                    Engine              Engineering      2
5                psychology               Psychology      2
38                    Econs                Economics      1
31                     Benl                     BENL      1
32            Biotechnology            Biotechnology      1
33  Business Administration  Business Administration      1
34                      CTS                      CTS      1
35           Communication             Communication      1
36             DIPLOMA TESL             

In [212]:
after_counts

,Course,Count
0,Engineering,20
1,BCS,18
2,BIT,10
3,KOE,6
4,Islamic Education,5
5,Biomedical Science,4
6,Psychology,3
7,BENL,3
8,Law,3
9,Fiqh & Fatwa,2


In [254]:
def clean_course(s):
    if pd.isna(s):
        return np.nan
    t = str(s).strip().lower()

    # remove punctuation & normalize spacing
    t_norm = re.sub(r'[^a-z]+', ' ', t).strip()

    # ✅ FIX: catch all Pendidikan Islam typos (pendidikan, pendidikar, pendidikann...)
    if t_norm.startswith("pendidik"):
        return "Islamic Education"
    if t_norm.startswith("math"):
        return "Mathematics"
    if t_norm in {"engin", "engine", "engineering"}:
        return "Engineering"
    if t_norm in {"law", "laws"}:
        return "Law"
    if t_norm in {"econs", "economics"}:
        return "Economics"
    if t_norm == "islamic education":
        return "Islamic Education"
    if t_norm.startswith("fiqh"):
        return "Fiqh & Fatwa"
    if t_norm in {"biomedical science", "biomedical sciences"}:
        return "Biomedical Science"
    if t_norm == "psychology":
        return "Psychology"
    if t_norm in {"communication", "communications"}:
        return "Communications"

    # ✅ Acronyms → uppercase
    acronyms = {
        "bcs":"BCS", "bit":"BIT", "it":"IT",
        "koe":"KOE", "mhsc":"MHSC", "cts":"CTS",
        "benl":"BENL", "kirkhs":"KIRKHS", "kop":"KOP"
    }
    if t_norm in acronyms:
        return acronyms[t_norm]

    # ✅ Fallback: title case
    return " ".join(word.capitalize() for word in t_norm.split())


In [256]:
df["Course_Standardized"] = df["What is your course?"].apply(clean_course)

In [258]:
# Build mapping to inspect
mapping = (df[[course_col, "Course_Standardized"]]
           .value_counts()
           .reset_index(name="Count")
           .sort_values("Count", ascending=False))

# Counts after
after_counts = (df["Course_Standardized"]
                .value_counts()
                .rename_axis("Course")
                .reset_index(name="Count"))

In [260]:
after_counts

,Course,Count
0,Engineering,20
1,BCS,18
2,BIT,10
3,KOE,6
4,Islamic Education,5
5,Biomedical Science,4
6,Psychology,3
7,BENL,3
8,Law,3
9,Fiqh & Fatwa,2


In [271]:
df

,Timestamp,Choose your gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic Education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Law,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathematics,year 4,3.00 - 3.49,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...
96,13/07/2020 19:56:49,Female,21.0,BCS,year 1,3.50 - 4.00,No,No,Yes,No,No
97,13/07/2020 21:21:42,Male,18.0,Engineering,Year 2,3.00 - 3.49,No,Yes,Yes,No,No
98,13/07/2020 21:22:56,Female,19.0,Nursing,Year 3,3.50 - 4.00,Yes,Yes,No,Yes,No
99,13/07/2020 21:23:57,Female,23.0,Islamic Education,year 4,3.50 - 4.00,No,No,No,No,No


In [273]:
df.to_csv("C:/Users/Jernie/OneDrive/Desktop/GITHUB PROJECTS/EDUCATION DATASET/New_Student_Mental_Health_Cleaned.csv", index=False)
